In [1]:
# %%
import os, pickle, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from scipy.stats import wilcoxon
from catboost import CatBoostRegressor, Pool

warnings.filterwarnings('ignore')

# ---- пути ----
BASE_PATH = "/Users/egorilin/Desktop/COMA"
BASE = Path(BASE_PATH)
ANALYSIS_DIR = BASE / 'analysis'
ANALYSIS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Output: {ANALYSIS_DIR}')

# ---- константы домена ----
TARGET_COLUMNS = [
    'S. aureus ATCC 43300 Activity MIC, мг/л',
    'S. aureus ATCC 43300 Activity MBC, мг/л',
    'E. coli ATCC 25922 Activity MIC, мг/л',
    'E. coli ATCC 25922 Activity MBC, мг/л',
]
SHORT_NAMES = ['SA_MIC', 'SA_MBC', 'EC_MIC', 'EC_MBC']
ORIG_COL_MAP = {sn: TARGET_COLUMNS[i] for i, sn in enumerate(SHORT_NAMES)}
OOD_COL_MAP = {
    'SA_MIC': 'MIC Sa',
    'SA_MBC': 'MBC Sa',
    'EC_MIC': 'MIC Ec',
    'EC_MBC': 'MBC Ec',
}

Device: cpu
Output: /Users/egorilin/Desktop/COMA/analysis


In [2]:
# %%
class AdvancedContrastiveEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dims=[512, 384, 256],
                 projection_dim=128, dropout=0.3):
        super().__init__()
        layers = []; prev = input_dim
        for hd in hidden_dims:
            layers.extend([nn.Linear(prev, hd), nn.BatchNorm1d(hd),
                           nn.ReLU(), nn.Dropout(dropout)])
            prev = hd
        self.encoder = nn.Sequential(*layers)
        self.projection_head = nn.Sequential(
            nn.Linear(hidden_dims[-1], projection_dim), nn.BatchNorm1d(projection_dim),
            nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(projection_dim, projection_dim), nn.BatchNorm1d(projection_dim),
        )

    def forward(self, x, return_projection=True):
        f = self.encoder(x)
        if return_projection:
            return f, self.projection_head(f)
        return f


class HardNegContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.05, base_temperature=0.07, alpha=0.3):
        super().__init__()
        self.t, self.bt, self.alpha = temperature, base_temperature, alpha
    def forward(self, features, labels):
        dev = features.device; bs = features.shape[0]
        f = F.normalize(features, dim=1)
        sim = f @ f.T / self.t
        labels = labels.view(-1, 1)
        mask_pos = torch.eq(labels, labels.T).float().to(dev)
        mask_neg = 1 - mask_pos
        logits_mask = torch.ones_like(mask_pos); logits_mask.fill_diagonal_(0)
        mask_pos *= logits_mask; mask_neg *= logits_mask
        neg = torch.exp(sim) * mask_neg
        hard_w = F.softmax(neg / self.t, dim=1)
        exp_l = torch.exp(sim) * logits_mask
        log_p = sim - torch.log(exp_l.sum(1, keepdim=True))
        mlp = (mask_pos * log_p).sum(1) / mask_pos.sum(1).clamp(min=1.0)
        pen = (hard_w * sim * mask_neg).sum(1)
        return (-(self.t / self.bt) * mlp + self.alpha * pen).mean()


class EmbDS(Dataset):
    def __init__(self, X, y, augment=True, noise=0.02):
        self.X = torch.FloatTensor(X); self.y = torch.LongTensor(y)
        self.aug, self.noise = augment, noise
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        x = self.X[i].clone()
        if self.aug and torch.rand(1).item() > 0.5:
            x = x + torch.randn_like(x) * self.noise
        return x, self.y[i]

print('✓ Архитектура и loss определены')

✓ Архитектура и loss определены


In [ ]:
# %%
from tqdm.auto import tqdm

with tqdm(total=6, desc='Loading artifacts') as pbar:
    # In-domain
    pbar.set_postfix_str('Test_molformer_original_biocides.xlsx')
    data_orig = pd.read_excel(f'{BASE_PATH}/Test_molformer_original_biocides.xlsx').drop(['Unnamed: 0'], axis=1)
    pbar.update(1)

    pbar.set_postfix_str('Data_biocides.xlsx')
    target_orig = pd.read_excel(f'{BASE_PATH}/Data_biocides.xlsx')
    pbar.update(1)

    # OOD
    pbar.set_postfix_str('QACs_SMILES_MIC_MBC.xlsx')
    ood_raw = pd.read_excel(f'{BASE_PATH}/QACs_SMILES_MIC_MBC.xlsx')
    pbar.update(1)

    pbar.set_postfix_str('Test_molformer_ood.xlsx')
    ood_molformer = pd.read_excel(f'{BASE_PATH}/Test_molformer_ood.xlsx')
    for c in ['Unnamed: 0']:
        if c in ood_molformer.columns:
            ood_molformer = ood_molformer.drop(columns=[c])
    ood_molformer = ood_molformer.select_dtypes(include=[np.number])
    ood_raw['MBC Ec'] = pd.to_numeric(ood_raw['MBC Ec'], errors='coerce')
    pbar.update(1)

    # Scaler + cls-trained encoder
    pbar.set_postfix_str('scaler_classification.pkl')
    with open(f'{BASE_PATH}/scaler_classification.pkl', 'rb') as f:
        scaler_clf = pickle.load(f)
    pbar.update(1)

    pbar.set_postfix_str('encoder_borderline_smote.pth')
    encoder_cls = AdvancedContrastiveEncoder(
        input_dim=768, hidden_dims=[512, 384, 256],
        projection_dim=128, dropout=0.3,
    ).to(device)
    encoder_cls.load_state_dict(
        torch.load(f'{BASE_PATH}/encoder_borderline_smote.pth', map_location=device)
    )
    encoder_cls.eval()
    pbar.update(1)


# Эмбеддинги обоих сплитов через cls-encoder (батчами с прогресс-баром)
def extract_emb(enc, X_scaled, batch_size=128, desc='extracting'):
    enc.eval()
    X_t = torch.FloatTensor(X_scaled)
    out = []
    with torch.no_grad():
        for i in tqdm(range(0, len(X_t), batch_size), desc=desc, leave=False):
            xb = X_t[i:i+batch_size].to(device)
            out.append(enc(xb, return_projection=False).cpu().numpy())
    return np.concatenate(out, axis=0)


orig_scaled = scaler_clf.transform(data_orig)
ood_scaled  = scaler_clf.transform(ood_molformer)

orig_emb_cls = extract_emb(encoder_cls, orig_scaled, desc='emb orig')
ood_emb_cls  = extract_emb(encoder_cls, ood_scaled,  desc='emb ood')

molformer_orig = data_orig.values.astype(float)
molformer_ood  = ood_molformer.values.astype(float)

print(f'data_orig:    {data_orig.shape}')
print(f'ood_raw:      {ood_raw.shape}')
print(f'orig_emb_cls: {orig_emb_cls.shape}')
print(f'ood_emb_cls:  {ood_emb_cls.shape}')

Loading artifacts:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
# %%
def build_pairs(mol_feat, target_df, col_map):
    rows = []
    for ti, sn in enumerate(SHORT_NAMES):
        col = col_map[sn]
        if col not in target_df.columns: continue
        vals = pd.to_numeric(target_df[col], errors='coerce')
        idx = np.where(vals.notna().values)[0]
        for gi in idx:
            y = float(vals.values[gi])
            if y <= 0: continue
            rows.append({
                'task': ti, 'sn': sn, 'gi': int(gi),
                'y_log2': float(np.log2(y)), 'y_raw': y,
                'mol_feat': mol_feat[gi],
            })
    return rows

def to_arrays(pairs):
    return {
        'mol':    np.stack([r['mol_feat'] for r in pairs]),
        'task':   np.array([r['task']     for r in pairs]),
        'gi':     np.array([r['gi']       for r in pairs]),
        'y_log2': np.array([r['y_log2']   for r in pairs]),
        'y_raw':  np.array([r['y_raw']    for r in pairs]),
    }

# Пары с MolFormer и emb_cls (одинаковый порядок строк, разные фичи)
A_in_mf  = to_arrays(build_pairs(molformer_orig, target_orig, ORIG_COL_MAP))
A_in_em  = to_arrays(build_pairs(orig_emb_cls,   target_orig, ORIG_COL_MAP))
A_ood_mf = to_arrays(build_pairs(molformer_ood,  ood_raw,    OOD_COL_MAP))
A_ood_em = to_arrays(build_pairs(ood_emb_cls,    ood_raw,    OOD_COL_MAP))

D_MF  = A_in_mf['mol'].shape[1]   # 768
D_EMB = A_in_em['mol'].shape[1]   # 256

# Combined фичи [molformer | emb] — для SHAP
X_in_combined  = np.hstack([A_in_mf['mol'],  A_in_em['mol']])
X_ood_combined = np.hstack([A_ood_mf['mol'], A_ood_em['mol']])

mf_cols  = [f'mf_{i}'  for i in range(D_MF)]
emb_cols = [f'emb_{i}' for i in range(D_EMB)]
ALL_COLS = mf_cols + emb_cols

print(f'in-domain pairs: {len(A_in_mf["task"])}')
print(f'OOD pairs:       {len(A_ood_mf["task"])}')
print(f'D_MF={D_MF}, D_EMB={D_EMB}, combined={X_in_combined.shape[1]}')

In [7]:
# %%
def make_mt_frame(X, task):
    df = pd.DataFrame(X, columns=ALL_COLS)
    df['task_id'] = task.astype(str)
    return df

df_in  = make_mt_frame(X_in_combined,  A_in_mf['task'])
df_ood = make_mt_frame(X_ood_combined, A_ood_mf['task'])

cb_mt = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                          verbose=False, random_seed=RANDOM_STATE)
cb_mt.fit(Pool(df_in, label=A_in_mf['y_log2'], cat_features=['task_id']))

print('Computing SHAP via CatBoost.get_feature_importance(type="ShapValues")...')
pool_in  = Pool(df_in,  cat_features=['task_id'])
pool_ood = Pool(df_ood, cat_features=['task_id'])
shap_in  = cb_mt.get_feature_importance(pool_in,  type='ShapValues')[:, :-1]
shap_ood = cb_mt.get_feature_importance(pool_ood, type='ShapValues')[:, :-1]
print(f'  shap_in:  {shap_in.shape}')
print(f'  shap_ood: {shap_ood.shape}')

slice_mf  = slice(0,    D_MF)
slice_emb = slice(D_MF, D_MF + D_EMB)
idx_task  = D_MF + D_EMB

def block_importance(s):
    a = np.abs(s)
    return {
        'molformer': a[:, slice_mf].sum(axis=1).mean(),
        'emb':       a[:, slice_emb].sum(axis=1).mean(),
        'task_id':   a[:, idx_task].mean(),
    }

block_in  = block_importance(shap_in)
block_ood = block_importance(shap_ood)
ratio_in  = block_in['emb']  / (block_in['molformer']  + 1e-12)
ratio_ood = block_ood['emb'] / (block_ood['molformer'] + 1e-12)

print('\n--- Block-level mean(|SHAP|) ---')
print(f'  IN-DOMAIN:  molformer={block_in["molformer"]:.4f}  '
      f'emb={block_in["emb"]:.4f}  task_id={block_in["task_id"]:.4f}')
print(f'  OOD:        molformer={block_ood["molformer"]:.4f}  '
      f'emb={block_ood["emb"]:.4f}  task_id={block_ood["task_id"]:.4f}')
print(f'\n  emb/molformer ratio:  in={ratio_in:.3f}  ood={ratio_ood:.3f}')

shap_summary_df = pd.DataFrame([
    {'split': 'in_domain', **block_in, 'emb_over_mf_ratio': ratio_in},
    {'split': 'ood',       **block_ood, 'emb_over_mf_ratio': ratio_ood},
])
shap_summary_df.to_csv(ANALYSIS_DIR / 'shap_block_importance.csv', index=False)

abs_ood = np.abs(shap_ood).mean(axis=0)
top_idx = np.argsort(abs_ood)[::-1][:20]
all_names = ALL_COLS + ['task_id']
top_df = pd.DataFrame({
    'feature': [all_names[i] for i in top_idx],
    'block':   ['molformer' if i < D_MF else ('emb' if i < D_MF + D_EMB else 'task_id')
                for i in top_idx],
    'mean_abs_shap_ood': abs_ood[top_idx],
})
print('\n--- TOP-20 features by |SHAP| on OOD ---')
print(top_df.to_string(index=False))
top_df.to_csv(ANALYSIS_DIR / 'shap_top20_features_ood.csv', index=False)
print(f'\n✓ {ANALYSIS_DIR / "shap_block_importance.csv"}')
print(f'✓ {ANALYSIS_DIR / "shap_top20_features_ood.csv"}')

NameError: name 'X_in_combined' is not defined

In [ ]:
# %%
def linear_cka(X, Y):
    Xc = X - X.mean(axis=0, keepdims=True)
    Yc = Y - Y.mean(axis=0, keepdims=True)
    dot = np.linalg.norm(Xc.T @ Yc, 'fro') ** 2
    nx  = np.linalg.norm(Xc.T @ Xc, 'fro')
    ny  = np.linalg.norm(Yc.T @ Yc, 'fro')
    return dot / (nx * ny + 1e-12)

def linear_probe_r2(X_src, X_tgt, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    r2s = []
    for tr, te in kf.split(X_src):
        sc = StandardScaler().fit(X_src[tr])
        Xtr = sc.transform(X_src[tr]); Xte = sc.transform(X_src[te])
        m = Ridge(alpha=1.0).fit(Xtr, X_tgt[tr])
        r2s.append(r2_score(X_tgt[te], m.predict(Xte),
                            multioutput='variance_weighted'))
    return float(np.mean(r2s)), float(np.std(r2s))

# уникальные молекулы (не пары)
_, idx_in  = np.unique(A_in_mf['gi'],  return_index=True)
_, idx_ood = np.unique(A_ood_mf['gi'], return_index=True)
mf_in_mol  = A_in_mf['mol'][idx_in];  em_in_mol  = A_in_em['mol'][idx_in]
mf_ood_mol = A_ood_mf['mol'][idx_ood]; em_ood_mol = A_ood_em['mol'][idx_ood]

cka_in  = linear_cka(mf_in_mol,  em_in_mol)
cka_ood = linear_cka(mf_ood_mol, em_ood_mol)
print(f'Linear CKA  in-domain: {cka_in:.4f}')
print(f'Linear CKA  OOD:       {cka_ood:.4f}')

r2_mf2emb_mean,  r2_mf2emb_std  = linear_probe_r2(mf_in_mol, em_in_mol)
r2_emb2mf_mean,  r2_emb2mf_std  = linear_probe_r2(em_in_mol, mf_in_mol)
print(f'\nlinear probe   molformer → emb_cls   R² = {r2_mf2emb_mean:.3f} ± {r2_mf2emb_std:.3f}')
print(f'linear probe   emb_cls → molformer   R² = {r2_emb2mf_mean:.3f} ± {r2_emb2mf_std:.3f}')

cka_df = pd.DataFrame([
    {'metric': 'linear_CKA',              'split': 'in_domain', 'value': cka_in},
    {'metric': 'linear_CKA',              'split': 'ood',       'value': cka_ood},
    {'metric': 'probe_R2_mf_to_emb_mean', 'split': 'in_domain', 'value': r2_mf2emb_mean},
    {'metric': 'probe_R2_mf_to_emb_std',  'split': 'in_domain', 'value': r2_mf2emb_std},
    {'metric': 'probe_R2_emb_to_mf_mean', 'split': 'in_domain', 'value': r2_emb2mf_mean},
    {'metric': 'probe_R2_emb_to_mf_std',  'split': 'in_domain', 'value': r2_emb2mf_std},
])
cka_df.to_csv(ANALYSIS_DIR / 'cka_probing.csv', index=False)
print(f'\n✓ {ANALYSIS_DIR / "cka_probing.csv"}')

print('\n--- Интерпретация probing ---')
if r2_mf2emb_mean > 0.95:
    print('  R²(mf→emb) > 0.95: emb почти полностью линейно восстановим из molformer (дистилляция)')
elif r2_mf2emb_mean > 0.80:
    print('  R²(mf→emb) ∈ (0.80, 0.95]: основная часть emb объясняется molformer, часть добавлена')
else:
    print('  R²(mf→emb) < 0.80: emb содержит существенно новую информацию')

In [ ]:
# %%
# Метка на молекулу: среднее log2(MIC/MBC) по доступным таргетам → 4 квантильных бина
all_y_log2 = []; all_gi = []
for ti, sn in enumerate(SHORT_NAMES):
    col = ORIG_COL_MAP[sn]
    vals = pd.to_numeric(target_orig[col], errors='coerce')
    idx = np.where(vals.notna().values)[0]
    for gi in idx:
        y = float(vals.values[gi])
        if y <= 0: continue
        all_y_log2.append(np.log2(y)); all_gi.append(gi)
all_y_log2 = np.array(all_y_log2); all_gi = np.array(all_gi)

mol_mean = pd.Series(all_y_log2).groupby(all_gi).mean()
bins = pd.qcut(mol_mean, q=4, labels=False, duplicates='drop')
mol_label = dict(zip(mol_mean.index, bins.astype(int).values))
train_mol_idx = np.array(sorted(mol_label.keys()))
X_train = data_orig.values[train_mol_idx].astype(np.float32)
y_train = np.array([mol_label[i] for i in train_mol_idx])
X_train_scaled = scaler_clf.transform(X_train)

print(f'N молекул для reg-encoder: {len(train_mol_idx)}')
print(f'Распределение бинов: {dict(zip(*np.unique(y_train, return_counts=True)))}')

# Обучение
encoder_reg = AdvancedContrastiveEncoder(input_dim=768, hidden_dims=[512, 384, 256],
                                         projection_dim=128, dropout=0.3).to(device)
loss_fn = HardNegContrastiveLoss(temperature=0.05, alpha=0.3)
opt = optim.AdamW(encoder_reg.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS = 50
sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

loader = DataLoader(EmbDS(X_train_scaled, y_train, augment=True),
                    batch_size=64, shuffle=True)
best = float('inf'); pat = 0; best_state = None
t0 = time.time()
for ep in range(EPOCHS):
    encoder_reg.train(); tot = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        _, proj = encoder_reg(xb, return_projection=True)
        l = loss_fn(proj, yb); l.backward()
        torch.nn.utils.clip_grad_norm_(encoder_reg.parameters(), 1.0)
        opt.step(); tot += l.item()
    avg = tot / len(loader); sch.step()
    if (ep+1) % 10 == 0: print(f'  ep {ep+1:02d}/{EPOCHS}  loss={avg:.4f}')
    if avg < best:
        best, pat = avg, 0
        best_state = {k: v.clone() for k, v in encoder_reg.state_dict().items()}
    else:
        pat += 1
        if pat >= 15: print(f'  early stop @ {ep+1}'); break
encoder_reg.load_state_dict(best_state)
torch.save(encoder_reg.state_dict(), ANALYSIS_DIR / 'encoder_reg_only.pth')
print(f'✓ reg-encoder обучен за {time.time()-t0:.1f}s, best_loss={best:.4f}')
print(f'✓ {ANALYSIS_DIR / "encoder_reg_only.pth"}')

# Извлекаем эмбеддинги
orig_emb_reg = extract_emb(encoder_reg, orig_scaled)
ood_emb_reg  = extract_emb(encoder_reg, ood_scaled)
print(f'orig_emb_reg: {orig_emb_reg.shape}, ood_emb_reg: {ood_emb_reg.shape}')

In [ ]:
# %%
def downstream_eval(mol_pairs_in, mol_pairs_ood, tag):
    df_tr = pd.DataFrame(mol_pairs_in,  columns=[f'f_{i}' for i in range(mol_pairs_in.shape[1])])
    df_tr['task_id'] = A_in_mf['task'].astype(str)
    df_te = pd.DataFrame(mol_pairs_ood, columns=df_tr.columns[:-1])
    df_te['task_id'] = A_ood_mf['task'].astype(str)
    cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6,
                           verbose=False, random_seed=RANDOM_STATE)
    cb.fit(Pool(df_tr, label=A_in_mf['y_log2'], cat_features=['task_id']))
    yp = cb.predict(Pool(df_te, cat_features=['task_id']))
    abs_res = np.abs(A_ood_mf['y_log2'] - yp)
    rmse = np.sqrt(np.mean((A_ood_mf['y_log2'] - yp) ** 2))
    mae  = np.mean(abs_res)
    r2   = r2_score(A_ood_mf['y_log2'], yp)
    print(f'  {tag:18s}  RMSE={rmse:.3f}  MAE={mae:.3f}  R²={r2:.3f}')
    return {'tag': tag, 'rmse': rmse, 'mae': mae, 'r2': r2,
            'abs_res': abs_res, 'y_pred': yp}

# Развёртываем эмбеддинги обратно по парам через gi
emb_reg_pairs_in  = orig_emb_reg[A_in_em['gi']]
emb_reg_pairs_ood = ood_emb_reg[A_ood_em['gi']]
emb_cls_pairs_in  = A_in_em['mol']
emb_cls_pairs_ood = A_ood_em['mol']
mf_pairs_in       = A_in_mf['mol']
mf_pairs_ood      = A_ood_mf['mol']

print('Downstream OOD (multi-task CatBoost, [feat | task_id])')
res_mf      = downstream_eval(mf_pairs_in,      mf_pairs_ood,      'molformer (768)')
res_emb_cls = downstream_eval(emb_cls_pairs_in, emb_cls_pairs_ood, 'emb_cls (256)')
res_emb_reg = downstream_eval(emb_reg_pairs_in, emb_reg_pairs_ood, 'emb_reg (256)')

def paired_test(a, b, name_a, name_b):
    diff = a - b
    try: stat, p = wilcoxon(diff, alternative='two-sided', zero_method='wilcox')
    except ValueError: stat, p = np.nan, np.nan
    print(f'  Δ|res| = ⟨{name_a}⟩−⟨{name_b}⟩ = {diff.mean():+.4f}   Wilcoxon p={p:.4g}')
    return diff.mean(), p

print('\nPaired Wilcoxon (per-molecule |residual|):')
d1, p1 = paired_test(res_emb_cls['abs_res'], res_emb_reg['abs_res'], 'emb_cls', 'emb_reg')
d2, p2 = paired_test(res_emb_cls['abs_res'], res_mf['abs_res'],      'emb_cls', 'molformer')
d3, p3 = paired_test(res_emb_reg['abs_res'], res_mf['abs_res'],      'emb_reg', 'molformer')

def boot_delta_rmse(a, b, B=2000, seed=42):
    rng = np.random.default_rng(seed); n = len(a); deltas = []
    for _ in range(B):
        idx = rng.integers(0, n, size=n)
        ra = np.sqrt(np.mean(a[idx] ** 2)); rb = np.sqrt(np.mean(b[idx] ** 2))
        deltas.append(ra - rb)
    deltas = np.array(deltas)
    return np.mean(deltas), np.percentile(deltas, [2.5, 97.5])

print('\nBootstrap 95% CI on ΔRMSE:')
ab_res = {'emb_cls': res_emb_cls['abs_res'], 'emb_reg': res_emb_reg['abs_res'],
          'molformer': res_mf['abs_res']}
boot_rows = []
for a, b in [('emb_cls','emb_reg'), ('emb_cls','molformer'), ('emb_reg','molformer')]:
    m, ci = boot_delta_rmse(ab_res[a], ab_res[b])
    print(f'  ΔRMSE({a} − {b}) = {m:+.4f}   95% CI = [{ci[0]:+.4f}, {ci[1]:+.4f}]')
    boot_rows.append({'a': a, 'b': b, 'mean_delta_rmse': m, 'ci_lo': ci[0], 'ci_hi': ci[1]})

ablation_df = pd.DataFrame([
    {'rep': 'molformer', 'rmse': res_mf['rmse'],      'mae': res_mf['mae'],      'r2': res_mf['r2']},
    {'rep': 'emb_cls',   'rmse': res_emb_cls['rmse'], 'mae': res_emb_cls['mae'], 'r2': res_emb_cls['r2']},
    {'rep': 'emb_reg',   'rmse': res_emb_reg['rmse'], 'mae': res_emb_reg['mae'], 'r2': res_emb_reg['r2']},
])
ablation_df.to_csv(ANALYSIS_DIR / 'ablation_downstream.csv', index=False)
pd.DataFrame(boot_rows).to_csv(ANALYSIS_DIR / 'ablation_bootstrap.csv', index=False)

# Per-target
per_t = []
print('\n--- Per-target RMSE на OOD ---')
for ti, sn in enumerate(SHORT_NAMES):
    m = A_ood_mf['task'] == ti
    if m.sum() < 2: continue
    rmse_mf  = np.sqrt(np.mean(ab_res['molformer'][m] ** 2))
    rmse_cls = np.sqrt(np.mean(ab_res['emb_cls'][m]   ** 2))
    rmse_reg = np.sqrt(np.mean(ab_res['emb_reg'][m]   ** 2))
    print(f'  {sn}:  molformer={rmse_mf:.3f}  emb_cls={rmse_cls:.3f}  emb_reg={rmse_reg:.3f}')
    per_t.append({'target': sn, 'molformer': rmse_mf, 'emb_cls': rmse_cls, 'emb_reg': rmse_reg})
pd.DataFrame(per_t).to_csv(ANALYSIS_DIR / 'ablation_per_target.csv', index=False)
print(f'\n✓ {ANALYSIS_DIR / "ablation_downstream.csv"}')
print(f'✓ {ANALYSIS_DIR / "ablation_bootstrap.csv"}')
print(f'✓ {ANALYSIS_DIR / "ablation_per_target.csv"}')

In [ ]:
# %%
print('=' * 72); print('ИТОГОВЫЙ ВЕРДИКТ'); print('=' * 72)

print('\n[1] Энкодер дистиллирует MolFormer или добавляет информацию?')
print(f'    CKA(mf, emb_cls) = {cka_in:.3f}  (1.0 = идеальная дистилляция)')
print(f'    Linear R²(mf → emb_cls) = {r2_mf2emb_mean:.3f}')
if r2_mf2emb_mean > 0.90:
    verdict1 = 'ДИСТИЛЛЯЦИЯ: emb почти полностью линейно восстановим из molformer.'
elif r2_mf2emb_mean > 0.70:
    verdict1 = 'ЧАСТИЧНОЕ ОБОГАЩЕНИЕ: основная часть выводится из molformer, часть добавлена.'
else:
    verdict1 = 'ОБОГАЩЕНИЕ: emb содержит существенно новую информацию.'
print(f'    → {verdict1}')

print(f'\n[2] SHAP — вклад emb-блока поверх molformer:')
print(f'    OOD: emb/molformer ratio = {ratio_ood:.3f}')
if ratio_ood < 0.10:
    verdict2 = 'emb-блок почти не используется деревом — molformer самодостаточен.'
elif ratio_ood < 0.40:
    verdict2 = 'emb даёт умеренный вклад поверх molformer.'
else:
    verdict2 = 'emb даёт значимый независимый вклад — энкодер добавил полезного сигнала.'
print(f'    → {verdict2}')

print('\n[3] Нужны ли данные классификатора, или хватит регрессионных?')
delta_rmse = res_emb_cls['rmse'] - res_emb_reg['rmse']
print(f'    RMSE(emb_cls) − RMSE(emb_reg) = {delta_rmse:+.4f}')
print(f'    Wilcoxon p (на |residuals|) = {p1:.4g}')
if p1 > 0.05:
    verdict3 = ('РАЗНИЦЫ НЕТ (p > 0.05): данные классификатора не дают значимого '
                'преимущества. Можно обучать энкодер только на регрессионных данных.')
elif delta_rmse < 0:
    verdict3 = 'CLS ЛУЧШЕ: данные классификатора дают значимое преимущество.'
else:
    verdict3 = 'REG ЛУЧШЕ: переключение на регрессионные данные улучшает энкодер.'
print(f'    → {verdict3}')

report = f"""
ИТОГОВЫЙ ВЕРДИКТ
================
[1] {verdict1}
    CKA(mf,emb_cls)={cka_in:.3f},  R²(mf→emb)={r2_mf2emb_mean:.3f}±{r2_mf2emb_std:.3f}

[2] {verdict2}
    OOD emb/mf SHAP ratio = {ratio_ood:.3f}
    OOD |SHAP|: mf={block_ood['molformer']:.4f}, emb={block_ood['emb']:.4f}

[3] {verdict3}
    OOD RMSE: molformer={res_mf['rmse']:.3f}, emb_cls={res_emb_cls['rmse']:.3f}, emb_reg={res_emb_reg['rmse']:.3f}
    Δ(cls−reg) RMSE = {delta_rmse:+.4f}, Wilcoxon p={p1:.4g}
"""
with open(ANALYSIS_DIR / 'VERDICT.txt', 'w', encoding='utf-8') as f:
    f.write(report)
print(f'\n✓ {ANALYSIS_DIR / "VERDICT.txt"}')

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
labels = ['in-domain', 'OOD']
mf_vals  = [block_in['molformer'], block_ood['molformer']]
emb_vals = [block_in['emb'],       block_ood['emb']]
x = np.arange(len(labels)); w = 0.35
ax.bar(x - w/2, mf_vals,  w, label='molformer (768)', color='#4C72B0', edgecolor='black')
ax.bar(x + w/2, emb_vals, w, label='emb (256)',      color='#55A868', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Block-level mean(|SHAP|)', fontweight='bold')
ax.set_title('SHAP block importance', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)

ax = axes[1]
bars = ['mf → emb_cls', 'emb_cls → mf']
vals = [r2_mf2emb_mean, r2_emb2mf_mean]
errs = [r2_mf2emb_std,  r2_emb2mf_std]
ax.bar(bars, vals, yerr=errs, capsize=8, color=['#DD8452', '#8172B3'], edgecolor='black')
ax.axhline(1.0, ls='--', c='gray', alpha=0.5)
ax.set_ylim(0, 1.05); ax.set_ylabel('Linear probe R²', fontweight='bold')
ax.set_title('emb ↔ molformer (linear probing)', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(vals): ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

ax = axes[2]
rmse_vals = [res_mf['rmse'], res_emb_cls['rmse'], res_emb_reg['rmse']]
ax.bar(['molformer', 'emb_cls', 'emb_reg'], rmse_vals,
       color=['#4C72B0', '#55A868', '#C44E52'], edgecolor='black')
ax.set_ylabel('OOD RMSE (log₂)', fontweight='bold')
ax.set_title('Ablation: cls- vs reg-trained encoder', fontweight='bold')
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(rmse_vals): ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'analysis_summary.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'\n✓ {ANALYSIS_DIR / "analysis_summary.png"}')
print('\n✨ ГОТОВО!')